# Livestock Feed Stockout-Risk Prediction (Kaggle)

**Purpose:** build a proof-of-concept classifier that estimates whether a feed ingredient may stock out within the next 7 days.

## Important limitation
No real farm inventory history with confirmed stockout labels has been supplied. This notebook therefore creates **simulated inventory scenarios** and labels them using a transparent forward simulation. The trained model learns patterns from that simulator; its metrics are **not real-farm performance** and must not be reported as validated field accuracy.

For real deployment, replace the simulated dataset with dated farm records containing inventory, usage/consumption, purchase orders, deliveries, supplier lead times, and confirmed stockout outcomes.

**Target:** `stockout_next_7d` (1 = stockout within 7 days, 0 = no stockout within 7 days).

In [ ]:
# 1. Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, average_precision_score, precision_recall_fscore_support
)
from sklearn.inspection import permutation_importance
import joblib, json, os, zipfile
from IPython.display import display, FileLink

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Imports ready")

## 2. Generate a labelled demonstration dataset

Each row represents an inventory check for one feed ingredient. The label is determined by simulating daily stock consumption and replenishment over the next 7 days, including supplier lead time and a reorder trigger. This is **simulated data**, not observed farm history.

In [ ]:
# 2. Simulate inventory snapshots and derive labels with a 7-day forward simulation
rng = np.random.default_rng(RANDOM_STATE)
N = 20000
ingredients = np.array(["maize", "wheat_grain", "bajra", "sorghum", "dry_fodder", "green_fodder"])
rows = []

for i in range(N):
    ingredient = rng.choice(ingredients)
    daily_demand = float(rng.uniform(5, 120))
    days_of_stock = float(rng.uniform(0.1, 35))
    current_stock = daily_demand * days_of_stock
    lead_time = int(rng.integers(1, 11))
    supplier_reliability = float(rng.uniform(0.65, 1.0))
    demand_volatility = float(rng.uniform(0.0, 0.45))
    price_change_pct = float(rng.uniform(-0.20, 0.50))
    open_purchase_qty = float(rng.choice([0, 0, 0, daily_demand*rng.uniform(2, 15)]))
    open_purchase_eta = int(rng.integers(1, 15)) if open_purchase_qty > 0 else 99
    reorder_point_days = float(rng.uniform(2, 14))
    recent_demand_change_pct = float(rng.uniform(-0.25, 0.55))
    expected_demand = daily_demand * (1 + recent_demand_change_pct)

    stock = current_stock
    stockout = False
    for day in range(1, 8):
        # Delivery arrives if it is scheduled within the forecast horizon.
        if open_purchase_qty > 0 and day == open_purchase_eta:
            stock += open_purchase_qty
        # Delivery delay risk is represented through a random late/missed delivery.
        stock -= max(0.1, rng.normal(expected_demand, expected_demand * demand_volatility))
        if stock < 0:
            stockout = True
            break
    rows.append({
        "ingredient": ingredient,
        "current_stock_kg": current_stock,
        "avg_daily_demand_kg": daily_demand,
        "days_of_stock": days_of_stock,
        "supplier_lead_time_days": lead_time,
        "supplier_reliability": supplier_reliability,
        "demand_volatility": demand_volatility,
        "price_change_pct": price_change_pct,
        "open_purchase_qty_kg": open_purchase_qty,
        "open_purchase_eta_days": open_purchase_eta if open_purchase_qty > 0 else 0,
        "reorder_point_days": reorder_point_days,
        "recent_demand_change_pct": recent_demand_change_pct,
        "stockout_next_7d": int(stockout)
    })

df = pd.DataFrame(rows)
print("Rows:", len(df))
print("Simulated stockout rate:", round(df["stockout_next_7d"].mean(), 3))
display(df.head())
display(df["stockout_next_7d"].value_counts().rename(index={0:"No stockout", 1:"Stockout"}))

## 3. Split data and train baseline + Random Forest

A stratified random split is used for this simulated demonstration. For real time-series farm records, use a **chronological split** so future dates never leak into training.

In [ ]:
target = "stockout_next_7d"
X = df.drop(columns=[target])
y = df[target]

categorical_features = ["ingredient"]
numeric_features = [c for c in X.columns if c not in categorical_features]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())
    ]), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])

baseline = Pipeline([
    ("prep", preprocessor),
    ("model", DummyClassifier(strategy="prior"))
])
rf = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

baseline.fit(X_train, y_train)
rf.fit(X_train, y_train)
print("Training complete")

In [ ]:
# 4. Evaluate
def evaluate_model(name, model, X_test, y_test):
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
    print(f"\\n{name}")
    print(classification_report(y_test, pred, digits=3, zero_division=0))
    if proba is not None and len(np.unique(y_test)) == 2:
        print("ROC-AUC:", round(roc_auc_score(y_test, proba), 4))
        print("PR-AUC / Average Precision:", round(average_precision_score(y_test, proba), 4))
    return pred, proba

base_pred, base_proba = evaluate_model("Dummy baseline", baseline, X_test, y_test)
rf_pred, rf_proba = evaluate_model("Random Forest", rf, X_test, y_test)

ConfusionMatrixDisplay.from_predictions(y_test, rf_pred)
plt.title("Random Forest — simulated test set")
plt.show()

## 5. Inspect the most influential inputs

Permutation importance estimates how much test-set performance changes when one feature is shuffled. Because the data and labels are simulated, these importances describe the simulation—not necessarily real farms.

In [ ]:
perm = permutation_importance(
    rf, X_test, y_test, n_repeats=5, random_state=RANDOM_STATE,
    scoring="average_precision", n_jobs=-1
)
importance = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)
display(importance)
importance.sort_values("importance_mean").plot(
    x="feature", y="importance_mean", kind="barh", legend=False,
    title="Permutation importance (simulated data)"
)
plt.xlabel("Decrease in average precision when shuffled")
plt.tight_layout()
plt.show()

## 6. Try a prediction for one inventory snapshot

Edit the values below to explore the prototype. The output is a **simulator-trained risk estimate**, not a validated operational probability.

In [ ]:
example = pd.DataFrame([{
    "ingredient": "maize",
    "current_stock_kg": 180.0,
    "avg_daily_demand_kg": 40.0,
    "days_of_stock": 4.5,
    "supplier_lead_time_days": 5,
    "supplier_reliability": 0.85,
    "demand_volatility": 0.20,
    "price_change_pct": 0.10,
    "open_purchase_qty_kg": 0.0,
    "open_purchase_eta_days": 0,
    "reorder_point_days": 7.0,
    "recent_demand_change_pct": 0.15
}])

risk = float(rf.predict_proba(example)[0, 1])
label = "HIGHER simulated risk" if risk >= 0.5 else "LOWER simulated risk"
print(f"Predicted probability under this simulator: {risk:.1%}")
print("Interpretation:", label)
print("Do not use this as a real farm risk probability until trained and validated on real labelled records.")

## 7. Save model and results

Artifacts include the trained model, simulated dataset, test predictions, evaluation summary and feature-importance table.

In [ ]:
os.makedirs("/kaggle/working/stockout_prediction_outputs", exist_ok=True)
out_dir = "/kaggle/working/stockout_prediction_outputs"

joblib.dump(rf, f"{out_dir}/stockout_risk_random_forest_SIMULATED.joblib")
df.to_csv(f"{out_dir}/simulated_stockout_training_data.csv", index=False)
predictions = X_test.copy()
predictions["actual_stockout_next_7d"] = y_test.values
predictions["predicted_stockout_next_7d"] = rf_pred
predictions["predicted_risk_score_simulated"] = rf_proba
predictions.to_csv(f"{out_dir}/test_predictions.csv", index=False)
importance.to_csv(f"{out_dir}/permutation_importance.csv", index=False)

metrics = {
    "dataset_type": "SIMULATED — not real farm observations",
    "rows": int(len(df)),
    "simulated_stockout_rate": float(df[target].mean()),
    "test_roc_auc": float(roc_auc_score(y_test, rf_proba)),
    "test_average_precision": float(average_precision_score(y_test, rf_proba)),
    "warning": "Metrics measure agreement with simulator-generated labels only; not real-world predictive performance."
}
with open(f"{out_dir}/metrics_and_limitations.json", "w") as f:
    json.dump(metrics, f, indent=2)

zip_path = "/kaggle/working/livestock_stockout_prediction_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for filename in os.listdir(out_dir):
        z.write(os.path.join(out_dir, filename), arcname=filename)

print("Saved:", zip_path)
display(FileLink(zip_path))

## 8. What is needed for a real stockout model?

Collect one row per ingredient per day, with at least:

- date and ingredient/feed type
- opening stock (kg)
- feed consumed or issued (kg)
- purchase quantity and order date
- expected and actual delivery dates
- supplier lead time / reliability
- confirmed stockout indicator, or enough data to derive stockout in the next 7 days

With real records, define the target from the actual inventory trajectory and validate on later dates. Do not mix synthetic and real records when reporting model metrics.